# Database Replication

Cross-region and cross-cloud **database replication** is the foundation for sharing data across regions.  
Snowflake supports:

| Capability | Description |
|---|---|
| **Database Replication** | Scheduled sync of a single database to one or more target accounts |
| **Replication Groups** | Manage replication of multiple objects (databases, shares, roles) as a unit |
| **Failover Groups** | Extend replication groups with automatic promotion for disaster recovery |

All replication is **asynchronous** — the target account receives a point-in-time consistent snapshot.  
Replication works across regions and cloud providers (AWS ↔ Azure ↔ GCP).

## 1. Setup — Source Database

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE DATABASE replication_demo;
CREATE SCHEMA replication_demo.analytics;

USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS repl_admin;
CREATE ROLE IF NOT EXISTS repl_monitor;
GRANT ROLE repl_admin TO ROLE SYSADMIN;
GRANT ROLE repl_monitor TO ROLE repl_admin;
GRANT USAGE ON DATABASE replication_demo TO ROLE repl_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE replication_demo TO ROLE repl_admin;
GRANT IMPORTED PRIVILEGES ON DATABASE snowflake TO ROLE repl_monitor;

USE ROLE ACCOUNTADMIN;
CREATE TABLE replication_demo.analytics.daily_sales AS
SELECT o_orderdate AS sale_date,
       COUNT(*) AS order_count,
       SUM(o_totalprice) AS total_revenue
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
GROUP BY o_orderdate
ORDER BY o_orderdate;

GRANT SELECT ON ALL TABLES IN SCHEMA replication_demo.analytics TO ROLE repl_admin;

## 2. Enable Replication

> **Note:** Replication requires a target account in a different region. Replace `<TARGET_ORG>.<TARGET_ACCOUNT>` with the actual target.

In [ ]:
-- Enable replication for the database to a target account
-- ALTER DATABASE replication_demo ENABLE REPLICATION TO ACCOUNTS <TARGET_ORG>.<TARGET_ACCOUNT>;

-- Verify replication is enabled
SHOW REPLICATION DATABASES;

## 3. Replication Groups

Replication groups allow managing multiple objects (databases, shares, roles) as a unit.

In [ ]:
-- Create a replication group (requires target account)
-- CREATE REPLICATION GROUP demo_repl_group
--     OBJECT_TYPES = DATABASES
--     ALLOWED_DATABASES = replication_demo
--     ALLOWED_ACCOUNTS = <TARGET_ORG>.<TARGET_ACCOUNT>
--     REPLICATION_SCHEDULE = '10 MINUTE';

-- Verify
SHOW REPLICATION GROUPS;

## 4. Failover Groups (DR)

Failover groups extend replication with automatic promotion capabilities.

In [ ]:
-- Create failover group
-- CREATE FAILOVER GROUP demo_failover_group
--     OBJECT_TYPES = DATABASES, ROLES
--     ALLOWED_DATABASES = replication_demo
--     ALLOWED_ACCOUNTS = <TARGET_ORG>.<TARGET_ACCOUNT>
--     REPLICATION_SCHEDULE = '10 MINUTE';

SHOW FAILOVER GROUPS;

## 5. Monitor Replication

In [ ]:
USE ROLE repl_monitor;

-- Replication history and costs
SELECT
    database_name,
    start_time,
    end_time,
    credits_used,
    bytes_transferred
FROM SNOWFLAKE.ACCOUNT_USAGE.REPLICATION_USAGE_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY start_time DESC;

-- Data transfer history (cross-region)
SELECT
    start_time,
    source_cloud,
    source_region,
    target_cloud,
    target_region,
    bytes_transferred,
    transfer_type
FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_TRANSFER_HISTORY
WHERE transfer_type = 'REPLICATION'
  AND start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY start_time DESC;

## 6. Consumer Side (Target Account)

These commands run in the **TARGET** account.

In [ ]:
-- In the TARGET account:
-- CREATE DATABASE replication_demo_replica
--     AS REPLICA OF <SOURCE_ORG>.<SOURCE_ACCOUNT>.replication_demo;

-- Refresh manually
-- ALTER DATABASE replication_demo_replica REFRESH;

-- Check refresh status
-- SELECT *
-- FROM TABLE(INFORMATION_SCHEMA.DATABASE_REFRESH_PROGRESS('replication_demo_replica'));

## 7. Teardown

In [ ]:
USE ROLE ACCOUNTADMIN;
-- DROP FAILOVER GROUP IF EXISTS demo_failover_group;
-- DROP REPLICATION GROUP IF EXISTS demo_repl_group;
DROP DATABASE IF EXISTS replication_demo;
DROP ROLE IF EXISTS repl_admin;
DROP ROLE IF EXISTS repl_monitor;

## References

- [Introduction to Replication and Failover](https://docs.snowflake.com/en/user-guide/account-replication-intro)
- [Replicating Databases Across Accounts](https://docs.snowflake.com/en/user-guide/database-replication-config)
- [CREATE REPLICATION GROUP](https://docs.snowflake.com/en/sql-reference/sql/create-replication-group)
- [CREATE FAILOVER GROUP](https://docs.snowflake.com/en/sql-reference/sql/create-failover-group)
- [ALTER DATABASE … ENABLE REPLICATION](https://docs.snowflake.com/en/sql-reference/sql/alter-database)
- [SHOW REPLICATION DATABASES](https://docs.snowflake.com/en/sql-reference/sql/show-replication-databases)
- [REPLICATION_USAGE_HISTORY View](https://docs.snowflake.com/en/sql-reference/account-usage/replication_usage_history)
- [DATA_TRANSFER_HISTORY View](https://docs.snowflake.com/en/sql-reference/account-usage/data_transfer_history)
- [Replication Cost](https://docs.snowflake.com/en/user-guide/cost-understanding-data-transfer)